# RS-VLM — Phase 2: Projector Alignment

Trains the **MLP projector** to bridge visual tokens (384-dim) into TinyLlama's embedding space (2048-dim).

- **What trains:** MLPProjector only  
- **What's frozen:** HybridEncoder (from Phase 1) + TinyLlama LLM  
- **Dataset:** RSICD (~10k images × 5 captions = ~50k image-caption pairs)  
- **Objective:** causal LM loss on caption tokens — teach the projector to speak LLM language  
- **Input:** `encoder_final.pt` from Phase 1  
- **Output:** `model_phase2_final.pt` → saved to Drive

## 0. Check GPU

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA:    {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:     {torch.cuda.get_device_name(0)}")
    print(f"VRAM:    {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. Mount Drive & verify

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

RSICD_DIR     = '/content/drive/MyDrive/rs_vlm_data/RSICD'
P1_CKPT       = '/content/drive/MyDrive/rs_vlm/checkpoints/phase1/encoder_final.pt'
P2_CKPT_DIR   = '/content/drive/MyDrive/rs_vlm/checkpoints/phase2'

# verify dataset is there
assert os.path.exists(RSICD_DIR),   f"RSICD not found at: {RSICD_DIR}"
assert os.path.exists(P1_CKPT),     f"Phase 1 checkpoint not found at: {P1_CKPT}"

# show RSICD structure
print("RSICD contents:")
for item in os.listdir(RSICD_DIR):
    full = os.path.join(RSICD_DIR, item)
    if os.path.isdir(full):
        print(f"  {item}/  ({len(os.listdir(full))} files)")
    else:
        size = os.path.getsize(full) / 1e6
        print(f"  {item}  ({size:.1f} MB)")

os.makedirs(P2_CKPT_DIR, exist_ok=True)
print(f"\nPhase 1 checkpoint: {P1_CKPT}")
print(f"Phase 2 output dir: {P2_CKPT_DIR}")

## 2. Clone / pull repo & install deps

In [ ]:
import os

if not os.path.exists('/content/rs_vlm'):
    !git clone https://github.com/sharksurfauto-byte/rs_vlm.git /content/rs_vlm
else:
    !git -C /content/rs_vlm pull
    # clear pycache so Python reloads updated modules
    !find /content/rs_vlm -name '*.pyc' -delete
    !find /content/rs_vlm -name '__pycache__' -type d -exec rm -rf {} + 2>/dev/null; true

%cd /content/rs_vlm
!pip install -q -r requirements.txt
print('Ready.')

## 3. Copy RSICD to local disk

Same trick as Phase 1 — Drive reads are slow, local disk is 10-50× faster.

In [ ]:
import shutil, os, time

SRC  = '/content/drive/MyDrive/rs_vlm_data/RSICD'
DEST = '/content/RSICD'

if not os.path.exists(DEST):
    print("Copying RSICD to local disk...")
    t = time.time()
    shutil.copytree(SRC, DEST)
    print(f"Done in {time.time()-t:.0f}s")
else:
    print("RSICD already on local disk.")

# quick sanity check
imgs = os.path.join(DEST, 'images')
json_file = os.path.join(DEST, 'dataset_rsicd.json')
print(f"images/ : {len(os.listdir(imgs))} files")
print(f"json    : {os.path.exists(json_file)}")

## 4. Patch config

In [ ]:
import yaml, sys
sys.path.insert(0, '/content/rs_vlm')

with open('/content/rs_vlm/configs/colab_config.yaml', 'r') as f:
    cfg = yaml.safe_load(f)

cfg['data']['rsicd_root']      = '/content/RSICD'   # local disk
cfg['data']['num_workers']     = 2
cfg['phase2']['batch_size']    = 8                  # TinyLlama is heavy
cfg['phase2']['epochs']        = 10
cfg['phase2']['checkpoint_dir']= P2_CKPT_DIR

with open('/content/rs_vlm/configs/colab_config.yaml', 'w') as f:
    yaml.dump(cfg, f)

print(f"rsicd_root:    {cfg['data']['rsicd_root']}")
print(f"batch_size:    {cfg['phase2']['batch_size']}")
print(f"epochs:        {cfg['phase2']['epochs']}")
print(f"checkpoint_dir:{cfg['phase2']['checkpoint_dir']}")

## 5. Smoke test — dataloader + encoder loading

In [ ]:
import torch, sys
sys.path.insert(0, '/content/rs_vlm')

from data.rsicd import get_rsicd_dataloader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

loader = get_rsicd_dataloader(root='/content/RSICD', train=True, batch_size=4, num_workers=0)
batch  = next(iter(loader))

print(f"image:   {batch['image'].shape}")
print(f"gsd:     {batch['gsd']}")
print(f"caption: {batch['caption'][0][:80]}...")
print(f"Total train batches (bs=4): {len(loader)}")
print(f"≈ {len(loader.dataset):,} training caption-image pairs")

# also verify the Phase 1 checkpoint loads
ckpt = torch.load(P1_CKPT, map_location='cpu')
print(f"\nPhase 1 ckpt — epoch: {ckpt['epoch']}, loss: {ckpt['loss']:.4f}")
print("Dataloader + checkpoint smoke test passed!")

## 6. Enable gradient checkpointing on TinyLlama

**Why:** even though the LLM is frozen, gradients still flow *through* it back to the projector. Without checkpointing, PyTorch stores all 22 LLM layer activations → OOM on T4.  
Gradient checkpointing recomputes activations during backward instead of storing them — trades a bit of compute for ~50% VRAM savings.

In [ ]:
# this cell just defines the helper — actual model is built inside train_phase2
# we patch train_phase2 to enable grad checkpointing right after model init

import torch
from model.rs_vlm import RSVLM

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("Building model (downloading TinyLlama if not cached)...")
model = RSVLM(cnn_pretrained=True).to(device)

# load Phase 1 encoder weights
ckpt = torch.load(P1_CKPT, map_location=device)
model.encoder.load_state_dict(ckpt['model_state_dict'])
print(f"Phase 1 encoder loaded (epoch {ckpt['epoch']}, loss {ckpt['loss']:.4f})")

# enable gradient checkpointing — critical for T4
model.llm.gradient_checkpointing_enable()
print("Gradient checkpointing enabled on LLM")

# check VRAM after model load
allocated = torch.cuda.memory_allocated() / 1e9
reserved  = torch.cuda.memory_reserved() / 1e9
print(f"VRAM used: {allocated:.1f} GB allocated / {reserved:.1f} GB reserved")

## 7. Run Phase 2 training

In [ ]:
from training.phase2_projector import train_phase2

model = train_phase2(
    config_path='configs/colab_config.yaml',
    phase1_checkpoint=P1_CKPT,
    resume_from=None,  # set to a .pt path to resume a crashed run
)

## 8. Verify checkpoint

In [ ]:
import torch, os

ckpt_path = os.path.join(P2_CKPT_DIR, 'model_phase2_final.pt')
ckpt = torch.load(ckpt_path, map_location='cpu')

print(f"Saved at epoch : {ckpt['epoch']}")
print(f"Final loss     : {ckpt['loss']:.4f}")
print(f"Keys           : {list(ckpt.keys())}")
print(f"\nPath for Phase 3 → phase2_checkpoint='{ckpt_path}'")